# Predict Engine Research

Notebook-first CatBoost regression workflow for the prepared training dataset. The primary artifact is `output/model.cbm`.


In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "src":
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

project_root


PosixPath('/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research')

In [2]:
import json
from statistics import median

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, train_test_split

from models.config import default_model_params
from models.regressor import build_regressor, resolve_categorical_columns, resolve_feature_columns
from runtime.device import get_device_report
from tracking.logger import get_logger
from tracking.wandb_tracker import finish_run, log_metrics, start_run, update_summary


## Metric Recap

This notebook trains a **regression** model, not a classifier.

Primary metrics for this task:
- `MAE`: average absolute price error in `price_manwon`, easiest to interpret.
- `RMSE`: penalizes large misses more strongly than MAE.
- `R2`: how much variance the model explains.
- CatBoost train/validation loss is tracked with `RMSE` during fitting.

`precision`, `recall`, `F1`, and `accuracy` are classification metrics, so they are not the right primary metrics here.


In [3]:
data_path = project_root / "data" / "prepared_training.csv"

target_column = "price_manwon"
feature_columns = [
    "brand",
    "model_name",
    "trim_name",
    "major_category",
    "size_score",
    "vehicle_age_years",
    "color",
    "mileage_km",
]
categorical_columns = [
    "brand",
    "model_name",
    "trim_name",
    "major_category",
    "color",
]

drop_unknown_major_category = True
major_category_unknown_value = "unknown"

output_dir = project_root / "output"
model_path = output_dir / "model.cbm"
metrics_path = output_dir / "metrics.json"
feature_manifest_path = output_dir / "feature_manifest.json"

use_cross_validation = True
n_splits = 5
holdout_valid_size = 0.2
random_seed = 42
max_iterations = 2000
early_stopping_rounds = 300
train_log_period = 25
final_iteration_strategy = "p75"

use_wandb = True
wandb_project = "predict_engine_research"
wandb_entity = "wonbeenlee093-asdf"
wandb_run_name = None
wandb_tags = ["catboost", "prepared_training"]

model_params = default_model_params()
model_params["random_seed"] = random_seed
model_params["iterations"] = max_iterations
model_params["verbose"] = train_log_period
model_params["allow_writing_files"] = False

{
    "data_path": str(data_path),
    "target_column": target_column,
    "feature_columns": feature_columns,
    "categorical_columns": categorical_columns,
    "use_cross_validation": use_cross_validation,
    "n_splits": n_splits,
    "holdout_valid_size": holdout_valid_size,
    "max_iterations": max_iterations,
    "early_stopping_rounds": early_stopping_rounds,
    "train_log_period": train_log_period,
    "final_iteration_strategy": final_iteration_strategy,
    "model_params": model_params,
}


{'data_path': '/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/data/prepared_training.csv',
 'target_column': 'price_manwon',
 'feature_columns': ['brand',
  'model_name',
  'trim_name',
  'major_category',
  'size_score',
  'vehicle_age_years',
  'color',
  'mileage_km'],
 'categorical_columns': ['brand',
  'model_name',
  'trim_name',
  'major_category',
  'color'],
 'use_cross_validation': True,
 'n_splits': 5,
 'holdout_valid_size': 0.2,
 'max_iterations': 2000,
 'early_stopping_rounds': 300,
 'train_log_period': 25,
 'final_iteration_strategy': 'p75',
 'model_params': {'loss_function': 'RMSE',
  'eval_metric': 'RMSE',
  'iterations': 2000,
  'learning_rate': 0.05,
  'depth': 8,
  'verbose': 25,
  'allow_writing_files': False,
  'random_seed': 42}}

In [4]:
logger = get_logger()
device_report = get_device_report()
logger.info("Torch device report: %s", device_report)
device_report


2026-03-18 02:18:14 | INFO | Torch device report: {'torch_installed': True, 'torch_version': '2.10.0', 'preferred_device': 'mps', 'cuda_available': False, 'mps_available': True}


{'torch_installed': True,
 'torch_version': '2.10.0',
 'preferred_device': 'mps',
 'cuda_available': False,
 'mps_available': True}

In [5]:
frame = pd.read_csv(data_path)

if drop_unknown_major_category and "major_category" in frame.columns:
    row_count_before = len(frame)
    frame = frame.loc[
        frame["major_category"].fillna(major_category_unknown_value).ne(major_category_unknown_value)
    ].copy()
    logger.info(
        "Dropped %s rows where major_category == %r",
        row_count_before - len(frame),
        major_category_unknown_value,
    )

feature_columns = resolve_feature_columns(frame, target_column, feature_columns)
categorical_columns = resolve_categorical_columns(frame, feature_columns, categorical_columns)

leakage_columns = [column for column in [target_column] if column in feature_columns]
if leakage_columns:
    raise ValueError(f"target leakage detected in feature columns: {leakage_columns}")

for column in categorical_columns:
    frame[column] = frame[column].astype("string").fillna("__missing__")

X = frame.loc[:, feature_columns].copy()
for column in categorical_columns:
    X[column] = X[column].astype(str)

y = pd.to_numeric(frame[target_column], errors="raise")

logger.info("Loaded %s rows with %s features", len(frame), len(feature_columns))
logger.info("Detected categorical columns: %s", categorical_columns)
logger.info("Target column: %s", target_column)
frame.head()


2026-03-18 02:18:14 | INFO | Dropped 1 rows where major_category == 'unknown'
2026-03-18 02:18:14 | INFO | Loaded 2130 rows with 8 features
2026-03-18 02:18:14 | INFO | Detected categorical columns: ['brand', 'model_name', 'trim_name', 'major_category', 'color']
2026-03-18 02:18:14 | INFO | Target column: price_manwon


,brand,model_name,trim_name,major_category,size_score,model_year,vehicle_age_years,color,mileage_km,price_manwon
0,hyundai,캐스퍼,디 에센셜,suv,0.721501,2023,2.92,초록(연두),5770.0,1780
1,kia,올 뉴 모닝 [JA],프레스티지,hatchback,22.797927,2018,8.17,검정색,46161.0,890
2,kgm,코란도 스포츠,CX7 클럽,truck,32.317073,2015,11.33,회색,250598.0,390
3,hyundai,코나,모던 팝,suv,6.709957,2018,8.58,"파랑(남색,곤색)",140663.0,1150
4,hyundai,아반떼AD,밸류 플러스,sedan,8.033419,2018,8.50,흰색,117863.0,970


In [6]:
run = start_run(
    enabled=use_wandb,
    project=wandb_project,
    entity=wandb_entity,
    name=wandb_run_name,
    tags=wandb_tags,
)

if run is not None:
    run.config.update(
        {
            "data_path": str(data_path),
            "target_column": target_column,
            "feature_columns": feature_columns,
            "categorical_columns": categorical_columns,
            "use_cross_validation": use_cross_validation,
            "n_splits": n_splits,
            "holdout_valid_size": holdout_valid_size,
            "max_iterations": max_iterations,
            "early_stopping_rounds": early_stopping_rounds,
            "train_log_period": train_log_period,
            "final_iteration_strategy": final_iteration_strategy,
            "model_params": model_params,
        }
    )
    logger.info(
        "W&B run started: entity=%s | project=%s | run_id=%s | url=%s",
        getattr(run, "entity", None),
        getattr(run, "project", None),
        getattr(run, "id", None),
        getattr(run, "url", None),
    )
else:
    logger.info("W&B logging disabled for this run")

run


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/iwonbin/.netrc
wandb: Currently logged in as: wonbeenlee093 (wonbeenlee093-asdf) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


2026-03-18 02:18:19 | INFO | W&B run started: entity=wonbeenlee093-asdf | project=predict_engine_research | run_id=nqz9y77k | url=https://wandb.ai/wonbeenlee093-asdf/predict_engine_research/runs/nqz9y77k


In [7]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse_value = float(root_mean_squared_error(y_true, y_pred))
    return {
        "mae_price_manwon": float(mean_absolute_error(y_true, y_pred)),
        "rmse_price_manwon": rmse_value,
        "r2_price_manwon": float(r2_score(y_true, y_pred)),
        "valid_loss": rmse_value,
    }


def extract_recommended_iterations(model):
    best_iteration = model.get_best_iteration()
    if best_iteration is not None and int(best_iteration) >= 0:
        return int(best_iteration) + 1
    return int(model.tree_count_)


def resolve_final_iterations(recommended_iterations, strategy):
    if not recommended_iterations:
        return int(model_params["iterations"])
    values = np.asarray(recommended_iterations, dtype=float)
    if strategy == "max":
        return int(values.max())
    if strategy == "p75":
        return int(round(np.percentile(values, 75)))
    return int(round(np.median(values)))


def log_eval_history(run, fold_index, evals_result, step_offset):
    if run is None:
        return 0

    learn_metrics = evals_result.get("learn", {})
    valid_metrics = evals_result.get("validation", {}) or evals_result.get("validation_0", {})
    metric_names = sorted(set(learn_metrics) | set(valid_metrics))
    if not metric_names:
        return 0

    history_length = max(
        [len(values) for values in list(learn_metrics.values()) + list(valid_metrics.values())],
        default=0,
    )
    for idx in range(history_length):
        payload = {"fold": fold_index, "iteration": idx + 1}
        for metric_name in metric_names:
            metric_key = metric_name.lower()
            if metric_key == "rmse":
                if idx < len(learn_metrics.get(metric_name, [])):
                    payload["train_loss"] = float(learn_metrics[metric_name][idx])
                if idx < len(valid_metrics.get(metric_name, [])):
                    payload["valid_loss"] = float(valid_metrics[metric_name][idx])
                continue
            if idx < len(learn_metrics.get(metric_name, [])):
                payload[f"learn_{metric_key}"] = float(learn_metrics[metric_name][idx])
            if idx < len(valid_metrics.get(metric_name, [])):
                payload[f"valid_{metric_key}"] = float(valid_metrics[metric_name][idx])
        log_metrics(run, payload, step=step_offset + idx + 1)
    return history_length


def build_split_iterator(X, y):
    if use_cross_validation:
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=random_seed)
        for fold_index, (train_index, valid_index) in enumerate(splitter.split(X, y), start=1):
            yield fold_index, train_index, valid_index, f"cv_fold_{fold_index}"
        return

    indices = np.arange(len(X))
    train_index, valid_index = train_test_split(
        indices,
        test_size=holdout_valid_size,
        random_state=random_seed,
        shuffle=True,
    )
    yield 1, train_index, valid_index, "holdout"


In [8]:
fold_results = []
recommended_iterations = []
iteration_step_offset = 0

for fold_index, train_index, valid_index, split_name in build_split_iterator(X, y):
    total_splits = n_splits if use_cross_validation else 1
    logger.info(
        "Starting %s (%s/%s) | max_iterations=%s | patience=%s",
        split_name,
        fold_index,
        total_splits,
        model_params["iterations"],
        early_stopping_rounds,
    )

    X_train = X.iloc[train_index]
    X_valid = X.iloc[valid_index]
    y_train = y.iloc[train_index]
    y_valid = y.iloc[valid_index]
    fold_model_params = dict(model_params)
    fold_model_params["use_best_model"] = True
    model = build_regressor(fold_model_params)

    fit_kwargs = {
        "eval_set": (X_valid, y_valid),
        "early_stopping_rounds": early_stopping_rounds,
    }
    if categorical_columns:
        fit_kwargs["cat_features"] = categorical_columns

    model.fit(X_train, y_train, **fit_kwargs)

    predictions = model.predict(X_valid)
    fold_metrics = {
        "fold": fold_index,
        "split_name": split_name,
        **compute_metrics(y_valid, predictions),
        "best_iteration": extract_recommended_iterations(model),
        "tree_count": int(model.tree_count_),
    }
    fold_results.append(fold_metrics)
    recommended_iterations.append(fold_metrics["best_iteration"])

    evals_result = model.get_evals_result()
    logged_history_rows = log_eval_history(run, fold_index, evals_result, iteration_step_offset)
    iteration_step_offset += logged_history_rows

    logger.info("%s metrics: %s", split_name, fold_metrics)
    log_metrics(
        run,
        {
            "fold": fold_index,
            "mae_price_manwon": fold_metrics["mae_price_manwon"],
            "rmse_price_manwon": fold_metrics["rmse_price_manwon"],
            "r2_price_manwon": fold_metrics["r2_price_manwon"],
            "valid_loss": fold_metrics["valid_loss"],
            "best_iteration": fold_metrics["best_iteration"],
            "tree_count": fold_metrics["tree_count"],
        },
        step=iteration_step_offset + fold_index,
    )

fold_results_df = pd.DataFrame(fold_results)
fold_results_df


2026-03-18 02:18:19 | INFO | Starting cv_fold_1 (1/5) | max_iterations=2000 | patience=300


0:	learn: 1320.1684487	test: 1286.4792726	best: 1286.4792726 (0)	total: 59.2ms	remaining: 1m 58s
25:	learn: 735.3350386	test: 707.6736377	best: 707.6736377 (25)	total: 129ms	remaining: 9.82s
50:	learn: 569.2866129	test: 561.1511125	best: 561.1511125 (50)	total: 198ms	remaining: 7.55s
75:	learn: 512.4509569	test: 527.4262613	best: 527.4262613 (75)	total: 265ms	remaining: 6.71s
100:	learn: 476.3550074	test: 516.9409376	best: 516.9409376 (100)	total: 339ms	remaining: 6.37s
125:	learn: 450.6867381	test: 511.8872161	best: 511.8872161 (125)	total: 396ms	remaining: 5.89s
150:	learn: 434.0183259	test: 509.5489423	best: 509.4202364 (149)	total: 465ms	remaining: 5.69s
175:	learn: 412.5963234	test: 509.4241303	best: 509.2491028 (156)	total: 563ms	remaining: 5.83s
200:	learn: 398.1212568	test: 509.0830555	best: 509.0830555 (200)	total: 638ms	remaining: 5.71s
225:	learn: 382.9917751	test: 508.6575096	best: 508.6575096 (225)	total: 711ms	remaining: 5.58s
250:	learn: 369.1149430	test: 508.2067098	bes

2026-03-18 02:18:21 | INFO | cv_fold_1 metrics: {'fold': 1, 'split_name': 'cv_fold_1', 'mae_price_manwon': 246.74388176067302, 'rmse_price_manwon': 504.1965694560558, 'r2_price_manwon': 0.8558498826113097, 'valid_loss': 504.1965694560558, 'best_iteration': 401, 'tree_count': 401}
2026-03-18 02:18:21 | INFO | Starting cv_fold_2 (2/5) | max_iterations=2000 | patience=300


700:	learn: 208.5204448	test: 508.0739993	best: 504.1965695 (400)	total: 2.09s	remaining: 3.88s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 504.1965695
bestIteration = 400

Shrink model to first 401 iterations.
0:	learn: 1323.2128200	test: 1264.6674865	best: 1264.6674865 (0)	total: 3.33ms	remaining: 6.65s
25:	learn: 747.4797186	test: 645.3373819	best: 645.3373819 (25)	total: 72.6ms	remaining: 5.51s
50:	learn: 581.9132417	test: 469.0991078	best: 469.0991078 (50)	total: 133ms	remaining: 5.09s
75:	learn: 521.8724169	test: 437.4224103	best: 437.4224103 (75)	total: 210ms	remaining: 5.31s
100:	learn: 489.1985028	test: 427.9395797	best: 427.9395797 (100)	total: 283ms	remaining: 5.32s
125:	learn: 463.6220682	test: 422.6575965	best: 422.6575965 (125)	total: 377ms	remaining: 5.6s
150:	learn: 443.9040347	test: 421.6207730	best: 421.6207730 (150)	total: 476ms	remaining: 5.83s
175:	learn: 427.3088976	test: 419.9604570	best: 418.5476310 (169)	total: 546ms	remaining: 5.66s
200:

2026-03-18 02:18:24 | INFO | cv_fold_2 metrics: {'fold': 2, 'split_name': 'cv_fold_2', 'mae_price_manwon': 265.8878996829161, 'rmse_price_manwon': 412.11536534550453, 'r2_price_manwon': 0.9007573711207585, 'valid_loss': 412.11536534550453, 'best_iteration': 478, 'tree_count': 478}
2026-03-18 02:18:24 | INFO | Starting cv_fold_3 (3/5) | max_iterations=2000 | patience=300


...

2026-03-18 02:18:29 | INFO | cv_fold_4 metrics: {'fold': 4, 'split_name': 'cv_fold_4', 'mae_price_manwon': 213.7747926599878, 'rmse_price_manwon': 342.6867268131587, 'r2_price_manwon': 0.9297920202030284, 'valid_loss': 342.6867268131587, 'best_iteration': 955, 'tree_count': 955}
2026-03-18 02:18:29 | INFO | Starting cv_fold_5 (5/5) | max_iterations=2000 | patience=300


Stopped by overfitting detector  (300 iterations wait)

bestTest = 342.6867268
bestIteration = 954

Shrink model to first 955 iterations.
0:	learn: 1339.0112082	test: 1203.4085498	best: 1203.4085498 (0)	total: 2.16ms	remaining: 4.32s
25:	learn: 751.1710460	test: 582.3038984	best: 582.3038984 (25)	total: 66ms	remaining: 5.01s
50:	learn: 593.3738922	test: 433.7841144	best: 433.7841144 (50)	total: 127ms	remaining: 4.87s
75:	learn: 529.3399744	test: 413.1857628	best: 411.6610061 (69)	total: 189ms	remaining: 4.79s
100:	learn: 503.6772973	test: 414.3649516	best: 408.9864036 (81)	total: 244ms	remaining: 4.58s
125:	learn: 487.0728411	test: 418.7789135	best: 408.9864036 (81)	total: 309ms	remaining: 4.6s
150:	learn: 472.8171806	test: 427.9482015	best: 408.9864036 (81)	total: 366ms	remaining: 4.48s
175:	learn: 456.0330900	test: 428.7681074	best: 408.9864036 (81)	total: 437ms	remaining: 4.53s
200:	learn: 441.0198669	test: 432.8989723	best: 408.9864036 (81)	total: 499ms	remaining: 4.47s
225:	learn:

2026-03-18 02:18:30 | INFO | cv_fold_5 metrics: {'fold': 5, 'split_name': 'cv_fold_5', 'mae_price_manwon': 274.8016453265512, 'rmse_price_manwon': 408.98640357258984, 'r2_price_manwon': 0.892768386166782, 'valid_loss': 408.98640357258984, 'best_iteration': 82, 'tree_count': 82}


375:	learn: 352.7364952	test: 454.1227291	best: 408.9864036 (81)	total: 978ms	remaining: 4.22s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 408.9864036
bestIteration = 81

Shrink model to first 82 iterations.


,fold,split_name,mae_price_manwon,rmse_price_manwon,r2_price_manwon,valid_loss,best_iteration,tree_count
0,1,cv_fold_1,246.743882,504.196569,0.855850,504.196569,401,401
1,2,cv_fold_2,265.887900,412.115365,0.900757,412.115365,478,478
2,3,cv_fold_3,277.596072,888.463711,0.680725,888.463711,496,496
3,4,cv_fold_4,213.774793,342.686727,0.929792,342.686727,955,955
4,5,cv_fold_5,274.801645,408.986404,0.892768,408.986404,82,82


In [9]:
aggregate_metrics = {
    "mae_price_manwon": float(fold_results_df["mae_price_manwon"].mean()),
    "rmse_price_manwon": float(fold_results_df["rmse_price_manwon"].mean()),
    "r2_price_manwon": float(fold_results_df["r2_price_manwon"].mean()),
    "valid_loss": float(fold_results_df["valid_loss"].mean()),
}
final_iterations = resolve_final_iterations(recommended_iterations, final_iteration_strategy)

logger.info("Aggregate validation metrics: %s", aggregate_metrics)
logger.info(
    "Recommended final iterations from validation runs: %s (strategy=%s)",
    final_iterations,
    final_iteration_strategy,
)

update_summary(
    run,
    {
        **aggregate_metrics,
        "final_iterations": final_iterations,
        "final_iteration_strategy": final_iteration_strategy,
        "training_rows": int(len(frame)),
        "feature_count": int(len(feature_columns)),
    },
)

{
    "aggregate_metrics": aggregate_metrics,
    "final_iterations": final_iterations,
    "final_iteration_strategy": final_iteration_strategy,
}


2026-03-18 02:18:31 | INFO | Aggregate validation metrics: {'mae_price_manwon': 255.76085820767508, 'rmse_price_manwon': 511.2897552789632, 'r2_price_manwon': 0.8519785246120449, 'valid_loss': 511.2897552789632}
2026-03-18 02:18:31 | INFO | Recommended final iterations from validation runs: 496 (strategy=p75)


{'aggregate_metrics': {'mae_price_manwon': 255.76085820767508,
  'rmse_price_manwon': 511.2897552789632,
  'r2_price_manwon': 0.8519785246120449,
  'valid_loss': 511.2897552789632},
 'final_iterations': 496,
 'final_iteration_strategy': 'p75'}

In [10]:
output_dir.mkdir(parents=True, exist_ok=True)

final_model_params = dict(model_params)
final_model_params["iterations"] = final_iterations
final_model_params.pop("use_best_model", None)
final_model = build_regressor(final_model_params)

final_fit_kwargs = {}
if categorical_columns:
    final_fit_kwargs["cat_features"] = categorical_columns

logger.info("Training final model on full dataset with iterations=%s", final_iterations)
final_model.fit(X, y, **final_fit_kwargs)
final_model.save_model(model_path)

metrics_payload = {
    "target_column": target_column,
    "use_cross_validation": use_cross_validation,
    "n_splits": n_splits if use_cross_validation else 1,
    "holdout_valid_size": holdout_valid_size,
    "max_iterations": max_iterations,
    "early_stopping_rounds": early_stopping_rounds,
    "train_log_period": train_log_period,
    "final_iteration_strategy": final_iteration_strategy,
    "model_params": final_model_params,
    "fold_results": fold_results,
    "aggregate_metrics": aggregate_metrics,
    "final_iterations": final_iterations,
    "tree_count": int(final_model.tree_count_),
    "training_rows": int(len(frame)),
}
feature_manifest = {
    "target_column": target_column,
    "feature_columns": feature_columns,
    "categorical_columns": categorical_columns,
}

metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
feature_manifest_path.write_text(json.dumps(feature_manifest, indent=2), encoding="utf-8")

update_summary(
    run,
    {
        "model_path": str(model_path),
        "tree_count": int(final_model.tree_count_),
        "final_iterations": final_iterations,
        "final_iteration_strategy": final_iteration_strategy,
    },
)
finish_run(run)

logger.info("Saved model to %s", model_path)
logger.info("Saved metrics to %s", metrics_path)
logger.info("Saved feature manifest to %s", feature_manifest_path)

{
    "model_path": str(model_path),
    "metrics_path": str(metrics_path),
    "feature_manifest_path": str(feature_manifest_path),
}


2026-03-18 02:18:31 | INFO | Training final model on full dataset with iterations=496


0:	learn: 1310.0674442	total: 3.33ms	remaining: 1.65s
25:	learn: 716.4590260	total: 75.5ms	remaining: 1.36s
50:	learn: 554.1885976	total: 153ms	remaining: 1.33s
75:	learn: 504.7547005	total: 224ms	remaining: 1.24s
100:	learn: 472.9745414	total: 295ms	remaining: 1.16s
125:	learn: 447.1820755	total: 357ms	remaining: 1.05s
150:	learn: 420.3497705	total: 423ms	remaining: 967ms
175:	learn: 396.1179916	total: 511ms	remaining: 930ms
200:	learn: 375.1094962	total: 590ms	remaining: 865ms
225:	learn: 357.8640864	total: 659ms	remaining: 787ms
250:	learn: 342.6446055	total: 729ms	remaining: 712ms
275:	learn: 328.7950086	total: 807ms	remaining: 644ms
300:	learn: 317.3424910	total: 890ms	remaining: 577ms
325:	learn: 306.7962239	total: 971ms	remaining: 506ms
350:	learn: 297.7000388	total: 1.05s	remaining: 433ms
375:	learn: 286.2649053	total: 1.11s	remaining: 356ms
400:	learn: 276.0123197	total: 1.18s	remaining: 280ms
425:	learn: 267.6882542	total: 1.26s	remaining: 208ms
450:	learn: 259.5324342	total:

best_iteration,▄▄▄█▁
fold,▁▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆▆███
iteration,▁▁▂▂▅▂▃▃▃▃▄▅▅▁▁▃▃▄▄▄▅▁▁▂▃▃▃▃▄▅▆▆▆▆▇██▂▂▃
mae_price_manwon,▅▇█▁█
r2_price_manwon,▆▇▁█▇
rmse_price_manwon,▃▂█▁▂
train_loss,▃▂▂▁█▃▃▃▃▃▂▂▂▁▁▃▂▂▂▂▁▁█▆▄▃▃▂▂▂▂▁▁▁▁▁▅▄▃▃
tree_count,▄▄▄█▁
valid_loss,▇▃▃▃▃▅▃▂▂▂▂▂▂▂▂███▇▇▇▇▇▇▅▂▁▁▁▁▁▁▁▁▁▁▁▁▄▂
best_iteration,82
feature_count,8


2026-03-18 02:18:34 | INFO | Saved model to /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/model.cbm
2026-03-18 02:18:34 | INFO | Saved metrics to /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/metrics.json
2026-03-18 02:18:34 | INFO | Saved feature manifest to /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/feature_manifest.json


{'model_path': '/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/model.cbm',
 'metrics_path': '/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/metrics.json',
 'feature_manifest_path': '/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/feature_manifest.json'}